# 🎈 Streamlit 기초 정리노트 (01~06 통합)

원본 6개 파일(`01_기본출력.py` ~ `06_차트.py`)의 핵심 내용을 한눈에 보기 쉽게 정리했습니다.
- 원본 파일은 수정하지 않았고, 이 노트북은 **별도로 새로 만든 요약본**입니다.
- 여러 파일에서 반복되던 코드(경로 설정, 데이터 로딩+캐싱, `bar_chart` 등)는 **한 번만** 정리하고, 이후 섹션에서는 "위와 동일 패턴"으로 언급만 합니다.
- ⚠️ Streamlit 코드는 `streamlit run 파일명.py`로 실행하는 앱이라 노트북 셀에서 그대로 실행되지는 않습니다. 이 노트북은 **복습/참고용 코드 정리**이며, 실제 실행은 원본 `.py` 파일로 하시면 됩니다.

## 0. 전체를 관통하는 핵심 개념

Streamlit의 가장 중요한 동작 원리부터 이해하면 나머지가 다 쉬워집니다.

> **위젯을 하나라도 건드리면(클릭·입력 등) 스크립트 전체가 위에서 아래로 다시 실행된다.**

이 특성 때문에 두 가지 문제가 생기고, 각각을 해결하는 도구가 따로 있습니다.

| 문제 | 해결 도구 | 파일 |
|---|---|---|
| 재실행될 때마다 **변수 값이 초기화**된다 (카운터가 안 늘어남) | `st.session_state` | 03 |
| 재실행될 때마다 **느린 작업(파일 읽기 등)을 반복**한다 | `st.cache_data` | 04 |

또 하나, 여러 파일(04, 05, 06)에서 공통으로 쓰인 경로 패턴이 있습니다.

```python
# streamlit run 은 '명령을 실행한 폴더'가 기준(cwd)이지, 스크립트 위치가 아니다.
# 그래서 상대경로('./../data')는 실행 위치에 따라 깨질 수 있다.
# __file__(현재 스크립트 경로) 기준 절대경로를 쓰면 어디서 실행해도 안전하다.
from pathlib import Path
CSV_PATH = Path(__file__).resolve().parents[1] / 'data' / 'netflix_titles.csv'
```

이 패턴은 아래 4장(캐싱)에서 처음 등장하고, 5장·6장에서도 그대로 재사용됩니다.

## 1. 기본 출력 — `01_기본출력.py`

Streamlit의 기본 출력 함수들을 모아 놓은 챕터입니다.

In [ ]:
import streamlit as st
import pandas as pd

# set_page_config 는 반드시 스크립트에서 '가장 먼저' 호출해야 한다
st.set_page_config(page_title='1기본', layout='wide')

st.title('01. 기본 출력')
st.header('헤더')
st.subheader('서브헤더')

# write 는 만능이다 - 문자열, 숫자, DataFrame, 그래프 등 타입을 알아서 판단해 보여준다
st.write('일반 텍스트입니다.')
st.write({'딕셔너리도': '표로 보여준다', 'key': 'value'})

st.divider()  # 가로 구분선

df = pd.DataFrame({'이름': ['지민', '수진', '태양'], '점수': [90, 85, 78]})

st.subheader('표 출력')
st.dataframe(df, use_container_width=True)  # 정렬·검색 가능한 동적 표
st.table(df)                                 # 움직이지 않는 정적 표

st.divider()

# metric: 대시보드용 숫자 카드. columns 로 가로 배치
col1, col2 = st.columns(2)
col1.metric('평균 점수', f"{df['점수'].mean():.1f}", delta='2.3')
col2.metric('최고 점수', df['점수'].max())

st.divider()

# 알림 상자 4종
st.info('정보 메시지')
st.success('성공 메시지')
st.warning('경고 메시지')
st.error('에러 메시지')

**핵심 정리**
- `st.dataframe` (동적, 정렬 가능) vs `st.table` (정적) — 상황에 맞게 선택
- `st.columns(n)`으로 가로 배치 후 `col.metric()` 호출 → 대시보드 카드
- 알림 상자 4종: `info` / `success` / `warning` / `error`

## 2. 위젯 — 입력값 받기 (`02_위젯.py`)

모든 위젯은 **사용자가 고른 값을 그대로 반환**합니다. 그 반환값을 변수에 담아 쓰면 됩니다.

In [ ]:
import streamlit as st

st.title('02. 위젯 - 입력값 받기')

name = st.text_input('이름을 입력하세요', value='홍길동')
age = st.slider('나이', min_value=0, max_value=100, value=25)
city = st.selectbox('도시', ['서울', '부산', '대구'])
hobbies = st.multiselect('취미', ['독서', '운동', '게임', '음악'])
agree = st.checkbox('약관에 동의합니다')
gender = st.radio('성별', ['남', '여'], horizontal=True)

st.divider()

# 위젯을 하나라도 건드리면 스크립트가 처음부터 재실행되고, 아래 내용이 새 값으로 다시 그려진다
st.subheader('입력 결과')
st.write(f'**{name}** / {age}세 / {city} / {gender}')
st.write('취미:', ', '.join(hobbies) if hobbies else '(없음)')

if not agree:
    st.warning('약관에 동의해야 제출할 수 있습니다.')
elif st.button('제출'):
    # button 은 '눌린 그 순간의 재실행'에서만 True, 다음 재실행에서는 다시 False로 돌아간다
    st.success(f'{name}님 제출 완료!')
    st.balloons()

**핵심 정리**
- 위젯 = 값을 반환하는 함수. `text_input / slider / selectbox / multiselect / checkbox / radio`
- ⚠️ `st.button()`은 특이 케이스: **눌린 순간의 재실행 한 번만** `True`이고, 곧바로 다시 `False`로 리셋된다.
  → 그래서 버튼으로 값을 "누적"시키려면 3장의 `session_state`가 꼭 필요하다.

## 3. session_state — 값 유지하기 (`03_session_state.py`)

0장에서 말한 '재실행마다 초기화되는 문제'를 해결하는 도구입니다.

In [ ]:
import streamlit as st

st.title('03. session_state - 값 유지하기')

# ----- 잘못된 예: 매 재실행마다 0으로 초기화되어 절대 늘지 않는다 -----
bad_count = 0
if st.button('❌ 잘못된 카운터'):
    bad_count += 1
st.write('잘못된 카운터:', bad_count, '← 아무리 눌러도 0 또는 1')

st.divider()

# ----- 올바른 예: session_state 에 보관하면 재실행돼도 살아남는다 -----
# 이 if 문이 '최초 1회만 초기화'를 보장한다 (재실행마다 다시 0으로 덮어쓰지 않음)
if 'count' not in st.session_state:
    st.session_state.count = 0

col1, col2 = st.columns(2)
if col1.button('⭕ 올바른 카운터 +1'):
    st.session_state.count += 1
if col2.button('초기화'):
    st.session_state.count = 0

st.write('올바른 카운터:', st.session_state.count, '← 계속 누적된다')

st.divider()

# 실전 패턴: 검색 결과(리스트)를 보관해두면 다른 위젯을 건드려도 사라지지 않는다
if 'history' not in st.session_state:
    st.session_state.history = []

keyword = st.text_input('검색어')
if st.button('검색') and keyword:
    st.session_state.history.append(keyword)

st.subheader('검색 기록')
st.write(st.session_state.history)

# 디버깅용: 현재 보관 중인 전체 상태 확인
with st.expander('session_state 전체 보기'):
    st.write(dict(st.session_state))

**핵심 정리 — 초기화 공식**
```python
if 'key' not in st.session_state:
    st.session_state.key = 초깃값
```
이 패턴만 기억하면 카운터·검색 기록·장바구니 등 어떤 상태든 유지할 수 있습니다.

## 4. 캐싱 — `st.cache_data` (`04_cache.py`)

0장의 두 번째 문제(느린 작업 반복 실행)를 해결합니다. 여기서 등장하는 **경로 패턴 + `load_data` 함수**는 5장·6장에서도 그대로 재사용되므로, 이 장에서 확실히 정리합니다.

In [ ]:
import streamlit as st
import pandas as pd
import time
from pathlib import Path

st.title('04. cache_data - 다시 읽지 않기')

# __file__ 기준 절대경로 (0장 참고) — 04, 05, 06 파일 공통 패턴
#   streamlit_basic/04_cache.py → parents[1] = 프로젝트 루트
CSV_PATH = Path(__file__).resolve().parents[1] / 'data' / 'netflix_titles.csv'


@st.cache_data
def load_data(path):
    """CSV 를 읽어 DataFrame 으로 돌려준다.

    @st.cache_data 덕분에 같은 path 로 호출되면
    실제 읽기는 최초 1회만 수행되고, 이후에는 저장된 결과를 그대로 반환한다.
    """
    time.sleep(2)          # 느린 작업(예: DB 조회, 큰 파일 로딩)을 흉내
    return pd.read_csv(path)


start = time.time()
df = load_data(CSV_PATH)
elapsed = time.time() - start

st.metric('로딩 시간', f'{elapsed:.2f} 초')
st.info('처음에는 2초 이상, 이후 재실행에서는 0초에 가깝게 나온다.')

st.dataframe(df.head(20), use_container_width=True)

# 슬라이더를 움직여도 재실행되지만, load_data 는 캐시 덕분에 다시 안 돈다
n = st.slider('표시할 행 수', 5, 50, 20)
st.dataframe(df.head(n), use_container_width=True)

if st.button('캐시 지우기'):
    st.cache_data.clear()
    st.rerun()   # 스크립트를 강제로 다시 실행

**핵심 정리**
- `@st.cache_data`를 함수에 붙이면, **같은 인자**로 호출될 때 실제 함수 내부 코드는 최초 1회만 실행되고 이후엔 저장된 결과를 재사용한다.
- `st.cache_data.clear()`로 캐시 전체를 비울 수 있다.
- `session_state` vs `cache_data` 비교
  - `session_state` → **사용자별로**, 재실행 간 "값 유지"
  - `cache_data` → **함수 결과를 저장**해서 "다시 계산/로딩 안 하기" (여러 사용자가 공유 가능)

## 5. 레이아웃 (`05_레이아웃.py`)

데이터 로딩(`@st.cache_data` + `CSV_PATH`)은 **4장과 동일한 패턴**이라 아래 코드에서는 생략하지 않고 그대로 두되, 새로 배우는 부분은 **사이드바 / columns / tabs / expander** 네 가지입니다.

In [ ]:
import streamlit as st
import pandas as pd
from pathlib import Path

st.set_page_config(page_title='레이아웃', layout='wide')
st.title('05. 레이아웃')

CSV_PATH = Path(__file__).resolve().parents[1] / 'data' / 'netflix_titles.csv'


@st.cache_data
def load_data():
    return pd.read_csv(CSV_PATH)


df = load_data()

# ---------- ① 사이드바: 필터 위젯을 옆으로 빼서 화면을 넓게 쓴다 ----------
with st.sidebar:
    st.header('🔍 필터')
    kind = st.selectbox('종류', ['전체'] + df['type'].dropna().unique().tolist())
    year_min, year_max = st.slider(
        '제작 연도',
        int(df['release_year'].min()), int(df['release_year'].max()),
        (2015, 2021),
    )

filtered = df[df['release_year'].between(year_min, year_max)]
if kind != '전체':
    filtered = filtered[filtered['type'] == kind]

# ---------- ② columns: 숫자 카드를 가로로 배치 ----------
c1, c2, c3 = st.columns(3)
c1.metric('전체 작품', f'{len(df):,}')
c2.metric('필터 결과', f'{len(filtered):,}')
c3.metric('비율', f'{len(filtered) / len(df) * 100:.1f}%')

st.divider()

# ---------- ③ tabs: 같은 데이터를 여러 관점으로 전환해서 보기 ----------
tab1, tab2, tab3 = st.tabs(['📋 목록', '📊 연도별', '🎬 장르'])

with tab1:
    st.dataframe(
        filtered[['type', 'title', 'release_year', 'listed_in']].head(100),
        use_container_width=True,
    )

with tab2:
    by_year = filtered['release_year'].value_counts().sort_index()
    st.bar_chart(by_year)   # ※ 6장에서 자세히 다룸

with tab3:
    genres = filtered['listed_in'].str.split(', ').explode()
    st.bar_chart(genres.value_counts().head(10))

# ---------- ④ expander: 기본은 접혀 있다가 클릭하면 펼쳐지는 영역 ----------
with st.expander('원본 데이터 정보'):
    st.write('행 x 열:', df.shape)
    st.write('컬럼:', list(df.columns))

**핵심 정리 — 레이아웃 4종 세트**

| 컴포넌트 | 용도 |
|---|---|
| `st.sidebar` (`with st.sidebar:`) | 필터 등 보조 컨트롤을 본문과 분리 |
| `st.columns(n)` | 요소를 가로로 나란히 배치 (카드형 metric에 자주 사용) |
| `st.tabs([...])` | 같은 데이터를 여러 관점(탭)으로 전환해서 보기 |
| `st.expander('제목')` | 기본은 접혀 있다가 클릭 시 펼쳐지는 영역 (부가정보 숨기기용) |

데이터 로딩은 4장과 완전히 동일한 `@st.cache_data` + `CSV_PATH` 패턴입니다.

## 6. 차트 (`06_차트.py`)

데이터 로딩은 4·5장과 동일 패턴이므로 아래에서는 **차트 부분만** 집중해서 봅니다. 핵심은 "**내장 차트(간단)** vs **matplotlib(세밀한 제어)**" 두 가지 방식 비교입니다.

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rc
from pathlib import Path

# 한글 폰트 설정 (matplotlib 은 기본 폰트로 한글이 깨지므로 별도 지정 필요)
rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

st.title('06. 차트')

CSV_PATH = Path(__file__).resolve().parents[1] / 'data' / 'netflix_titles.csv'


@st.cache_data
def load_data():
    return pd.read_csv(CSV_PATH)


df = load_data()
by_year = df[df['release_year'] >= 2010]['release_year'].value_counts().sort_index()

# ---------- 방법 1: streamlit 내장 차트 (간단, 옵션은 적음) ----------
st.subheader('내장 차트 - st.bar_chart')
st.bar_chart(by_year)

st.divider()

# ---------- 방법 2: matplotlib (세밀한 제어 가능) ----------
st.subheader('matplotlib - st.pyplot')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_year.index, by_year.values, color='#E50914')
ax.set_title('연도별 넷플릭스 콘텐츠 수')
ax.set_xlabel('제작 연도')
ax.set_ylabel('작품 수')
ax.grid(axis='y', alpha=0.3)

st.pyplot(fig)   # Figure 객체를 그대로 넘긴다

st.caption('matplotlib 은 제목·색·격자 등을 세밀하게 제어할 수 있다.')

**핵심 정리 — 차트 두 방식 비교**

| | `st.bar_chart` (내장) | `st.pyplot` (matplotlib) |
|---|---|---|
| 코드량 | 한 줄로 끝 | figure/axes 직접 설정 |
| 커스터마이징 | 제한적 | 제목·색·격자·폰트 등 자유자재 |
| 언제 쓰나 | 빠르게 데이터 흐름만 확인할 때 | 발표/리포트용으로 다듬을 때 |

한글이 깨지지 않으려면 matplotlib 사용 전 `rc('font', family='Malgun Gothic')` 설정이 필요합니다 (Windows 기준).

## 📌 전체 요약표

| 챕터 | 파일 | 핵심 API | 한 줄 요약 |
|---|---|---|---|
| 1 | `01_기본출력.py` | `write / dataframe / table / metric / info·success·warning·error` | 화면에 뭔가를 "그냥 찍어보는" 기본기 |
| 2 | `02_위젯.py` | `text_input / slider / selectbox / multiselect / checkbox / radio / button` | 사용자 입력값 받기, 위젯 = 값을 반환하는 함수 |
| 3 | `03_session_state.py` | `st.session_state` | 재실행돼도 값이 초기화되지 않게 "기억"시키기 |
| 4 | `04_cache.py` | `@st.cache_data` | 같은 입력이면 무거운 연산/로딩을 재실행하지 않기 |
| 5 | `05_레이아웃.py` | `sidebar / columns / tabs / expander` | 화면 구조 잡기 (필터·카드·탭·접이식 영역) |
| 6 | `06_차트.py` | `st.bar_chart / st.pyplot` | 빠른 내장 차트 vs 세밀한 matplotlib 차트 |

**기억할 한 문장:** *"위젯을 건드리면 스크립트가 처음부터 다시 실행된다"* — 이 사실 하나가 `session_state`(3장)와 `cache_data`(4장)가 왜 필요한지의 이유입니다.